In [6]:
# Jupyter Notebook: LA Dilation — patient‑disjoint temporal splits with task‑specific cutpoints

from __future__ import annotations
import os
from dataclasses import dataclass
from datetime import date
from pathlib import Path
from typing import Optional, Tuple

import numpy as np
import pandas as pd

# ---- Column names (edit if yours differ) ----
STUDY_COL   = "STUDY_REF"
PATIENT_COL = "PATIENT_ID"
DATE_COL    = "STUDY_DATE"         # parseable to date string
LABEL_COL   = "LA_DILATION"        # in labels CSV (0/1 or strings)

# Required views for LA dilation (toggle to match your schema)
REQUIRED_VIEW_COLS = ["PLAX", "PSAX_AV", "A4C", "A2C", "A3C", "SUBCOSTAL_4C"]

# If LABEL_COL uses strings, map here; else identity for 0/1 integers
LABEL_MAP = {
    "normal": 0, "no": 0, "absent": 0, 0: 0,
    "dilated": 1, "present": 1, "yes": 1, 1: 1
}

ERA_CUTOFF = date(2015, 6, 1)      # carried to outputs for later analyses


In [ ]:
# ==== Set your CSV paths here ====
studies_csv = "/path/to/studies.csv"      # must have STUDY_REF, PATIENT_ID, STUDY_DATE
labels_csv  = "/path/to/la_labels.csv"    # must have STUDY_REF, LA_DILATION
views_csv   = None                        # optional: e.g., "/path/to/views.csv" with required view flags

out_dir     = "./splits_la_dilation"      # output folder for split CSVs

# ==== Task-specific temporal cutpoints on eligible patients ====
train_q       = 0.70       # first 70% (by eligible-patient order) -> train
val_q         = 0.85       # next 15% -> val ; remainder -> test
min_test_all  = 2000       # require at least this many eligible test rows (set lower if needed)
min_test_pos  = 0          # require at least this many positives in test (0 disables)
